In [9]:
# CELL 1 — Install required libraries

!pip -q install groq gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.8 MB/s eta 0:00:00


In [10]:
# CELL 2 — Groq API setup

from getpass import getpass
from groq import Groq

GROQ_API_KEY = getpass("Enter your Groq API key: ")

client = Groq(api_key=GROQ_API_KEY)

print("✅ Groq client connected successfully!")

Enter your Groq API key: ··········
✅ Groq client connected successfully!


In [11]:
# CELL 3 — Test current Groq model

MODEL_NAME = "openai/gpt-oss-20b"

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Explain photosynthesis in one simple sentence for a school student."
        }
    ],
    temperature=0.3
)

print("✅ AI is working!")
print("\nModel:", MODEL_NAME)
print("\nResponse:")
print(response.choices[0].message.content)

✅ AI is working!

Model: openai/gpt-oss-20b

Response:
Plants turn sunlight, water, and carbon dioxide into food and oxygen.


In [12]:
# CELL 4 — Misconception Analysis Agent

def analyze_misconception(question, student_answer):

    prompt = f"""
You are the Misconception Analysis Agent in an educational AI system
called MisconceptionHunter.

Your job is NOT simply to say whether the student's answer is correct.

You must identify the possible conceptual misconception behind the student's
answer.

Analyze the following:

QUESTION:
{question}

STUDENT ANSWER:
{student_answer}

Return your response using exactly this structure:

VERDICT:
[Correct / Partially Correct / Incorrect]

DETECTED MISCONCEPTION:
[Clearly explain the misconception, or say "No misconception detected."]

WHY IT HAPPENED:
[Explain the likely reasoning mistake in simple language.]

CORRECT CONCEPT:
[Explain the correct concept simply.]

LEARNING TIP:
[Give one short tip that can help the student avoid this mistake.]

Do not insult or blame the student.
Use simple, encouraging language.
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "You are an expert educational AI tutor."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3
    )

    return response.choices[0].message.content

In [13]:
# CELL 5 — Test Misconception Analysis

question = """
Why does an object continue moving for some time after being pushed,
even when no one is continuously pushing it?
"""

student_answer = """
Because a force remains inside the object after we push it,
and that force keeps the object moving.
"""

result = analyze_misconception(question, student_answer)

print("🧠 MISCONCEPTIONHUNTER ANALYSIS")
print("=" * 60)
print(result)

🧠 MISCONCEPTIONHUNTER ANALYSIS
VERDICT:
Incorrect

DETECTED MISCONCEPTION:
The student believes that a force “remains inside” the object after the push and that this lingering force keeps the object moving. They are confusing the idea of a force with the property of inertia or the absence of a net external force.

WHY IT HAPPENED:
The student likely mixed up Newton’s first law (an object keeps its state of motion unless acted on by a net external force) with the notion that a force can stay inside an object after it’s applied. They may have thought of force as a kind of “internal energy” that persists, rather than an interaction that must be present to change motion.

CORRECT CONCEPT:
An object continues moving because there is no net external force acting on it (ignoring small forces like friction or air resistance). This is inertia: a property of mass that resists changes in motion. Only when an external force (e.g., friction, a wall, or a new push) acts does the motion change.

LEAR

In [14]:
!pip install -q -U google-genai

from google import genai
from getpass import getpass

GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

print("✅ Gemini client connected!")

Enter your Gemini API key: ··········
✅ Gemini client connected!


In [15]:
response = gemini_client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Reply with exactly: VERIFIER TEST PASSED"
)

print(response.text)

VERIFIER TEST PASSED


In [16]:
# CELL 6 — Gemini Verifier Agent

def verify_misconception(question, student_answer, analysis):

    prompt = f"""
You are the Verifier Agent in MisconceptionHunter.

Another AI agent analyzed a student's answer.
Your job is to independently verify whether the detected misconception
is reasonable and whether the explanation is scientifically correct.

Do NOT blindly agree with the Analysis Agent.
Compare the student's actual answer with the analysis and independently
check the underlying concept.

QUESTION:
{question}

STUDENT ANSWER:
{student_answer}

ANALYSIS AGENT OUTPUT:
{analysis}

Return exactly this format:

VERIFICATION:
[Confirmed / Partially Confirmed / Not Confirmed]

CONFIDENCE:
[High / Medium / Low]

VERIFIED MISCONCEPTION:
[State the final misconception clearly.]

CORRECTION:
[Give the correct concept in simple language.]

FEEDBACK:
[Give one short, encouraging sentence for the student.]
"""

    response = gemini_client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

In [17]:
# CELL 7 — Test Verifier Agent

verification = verify_misconception(
    question,
    student_answer,
    result
)

print("🔍 VERIFIER AGENT")
print("=" * 60)
print(verification)

🔍 VERIFIER AGENT
VERIFICATION:
Confirmed

CONFIDENCE:
High

VERIFIED MISCONCEPTION:
The student holds the misconception that a force is an entity that can be transferred into and stored inside an object, and that this "stored force" is what sustains the object's motion after the push stops.

CORRECTION:
A force is an interaction between objects, not a substance stored inside them. An object keeps moving because of inertia—its natural tendency to maintain its current state of motion—until an external force (like friction) acts on it.

FEEDBACK:
You've got the right idea that something keeps the object going, but remember that motion continues naturally through inertia rather than stored pushing power!


In [18]:
# CELL 8 — Personalized Challenge Generator

def generate_challenge(question, student_answer, verification):

    prompt = f"""
You are the Personalized Challenge Agent in MisconceptionHunter.

Your task is to create a short follow-up question that specifically tests
whether the student has understood the misconception identified by the
Verifier Agent.

ORIGINAL QUESTION:
{question}

STUDENT ANSWER:
{student_answer}

VERIFIER OUTPUT:
{verification}

Create ONE new challenge question.

The challenge must:
- Target the same misconception
- Be different from the original question
- Be slightly easier or similar in difficulty
- Test understanding rather than memorization
- Be suitable for a student
- NOT reveal the answer

Return exactly:

CHALLENGE QUESTION:
[Your question]

WHY THIS CHALLENGE:
[One short explanation of what misconception it tests.]
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "You are an expert personalized learning agent."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.5
    )

    return response.choices[0].message.content

In [19]:
# CELL 9 — Generate Personalized Challenge

challenge = generate_challenge(
    question,
    student_answer,
    verification
)

print("🎯 PERSONALIZED CHALLENGE")
print("=" * 60)
print(challenge)

🎯 PERSONALIZED CHALLENGE
CHALLENGE QUESTION:
Suppose you push a skateboard across a flat surface and then let go. The skateboard continues to glide for a few meters before it comes to rest. What is causing the skateboard to keep moving after you stop pushing it?

WHY THIS CHALLENGE:
It asks the student to explain the mechanism that sustains motion after the push, probing whether they mistakenly think a force is stored inside the skateboard.


In [20]:
# CELL 10 — MisconceptionHunter Gradio Interface

import gradio as gr

def misconception_hunter(question, student_answer):

    if not question.strip() or not student_answer.strip():
        return (
            "⚠️ Please enter both a question and your answer.",
            "",
            ""
        )

    # Agent 1 — Analyze
    analysis = analyze_misconception(
        question,
        student_answer
    )

    # Agent 2 — Verify
    verification = verify_misconception(
        question,
        student_answer,
        analysis
    )

    # Agent 3 — Personalized Challenge
    challenge = generate_challenge(
        question,
        student_answer,
        verification
    )

    return analysis, verification, challenge


# Create the interface
with gr.Blocks(
    title="MisconceptionHunter",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown(
        """
        # 🧠 MisconceptionHunter
        ### Don't just find the wrong answer. Find the wrong understanding.

        **Agentic AI for personalized learning**
        """
    )

    gr.Markdown(
        """
        Enter a question and your answer.
        Our AI agents will analyze your reasoning, verify the misconception,
        and generate a personalized challenge.
        """
    )

    with gr.Row():

        with gr.Column():

            question_input = gr.Textbox(
                label="📚 Question",
                placeholder="Enter the question here...",
                lines=4
            )

            answer_input = gr.Textbox(
                label="✍️ Your Answer",
                placeholder="Explain your answer here...",
                lines=6
            )

            analyze_button = gr.Button(
                "🔎 Hunt My Misconception",
                variant="primary"
            )

        with gr.Column():

            analysis_output = gr.Textbox(
                label="🤖 Analysis Agent",
                lines=10
            )

            verification_output = gr.Textbox(
                label="🔍 Verifier Agent",
                lines=10
            )

            challenge_output = gr.Textbox(
                label="🎯 Personalized Challenge",
                lines=8
            )

    analyze_button.click(
        fn=misconception_hunter,
        inputs=[
            question_input,
            answer_input
        ],
        outputs=[
            analysis_output,
            verification_output,
            challenge_output
        ]
    )

    gr.Markdown(
        """
        ---
        ### 🚀 Team Codenova
        **MisconceptionHunter | IMPACTX '26**

        *From correcting answers to correcting understanding.*
        """
    )


demo.launch(share=True)

/tmp/ipykernel_4144/2647778294.py:38: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c950e36be9b47db209.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [21]:
# CELL 11 — MisconceptionHunter Professional UI

import gradio as gr

def misconception_hunter_ui(question, student_answer):

    if not question.strip() or not student_answer.strip():
        return (
            "⚠️ Please enter both the question and your answer.",
            "",
            ""
        )

    try:
        # Analysis Agent
        analysis = analyze_misconception(
            question,
            student_answer
        )

        # Verifier Agent
        verification = verify_misconception(
            question,
            student_answer,
            analysis
        )

        # Challenge Agent
        challenge = generate_challenge(
            question,
            student_answer,
            verification
        )

        return analysis, verification, challenge

    except Exception as e:
        return (
            f"⚠️ Something went wrong:\n\n{str(e)}",
            "",
            ""
        )


custom_css = """
body {
    background: #0b1020;
}

.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
}

.hero {
    text-align: center;
    padding: 25px;
    border-radius: 20px;
    background: linear-gradient(135deg, #151b3d, #24205c);
    margin-bottom: 20px;
}

.hero h1 {
    font-size: 42px;
    margin-bottom: 5px;
}

.hero p {
    font-size: 18px;
}

.section-card {
    border-radius: 18px;
    padding: 10px;
}

.agent-title {
    font-size: 18px;
    font-weight: bold;
}
"""

with gr.Blocks(
    title="MisconceptionHunter",
    css=custom_css,
    theme=gr.themes.Soft(
        primary_hue="indigo",
        secondary_hue="blue"
    )
) as app:

    # HERO SECTION
    gr.HTML("""
    <div class="hero">
        <h1>🧠 MisconceptionHunter</h1>
        <p>
        Don't just find the wrong answer.
        <b>Find the wrong understanding.</b>
        </p>
        <p>🤖 Agentic AI • 🎓 Personalized Learning • 🔍 Misconception Detection</p>
    </div>
    """)

    gr.Markdown(
        """
        ### 🎯 How it works

        **Student Answer → Analysis Agent → Verifier Agent → Personalized Challenge**
        """
    )

    # INPUT AREA
    with gr.Row():

        with gr.Column(scale=1):

            gr.Markdown("### 📝 Student Input")

            question_input = gr.Textbox(
                label="Question",
                placeholder="Example: Why does an object continue moving after being pushed?",
                lines=5
            )

            answer_input = gr.Textbox(
                label="Your Answer",
                placeholder="Write your reasoning here...",
                lines=7
            )

            analyze_button = gr.Button(
                "🔎 HUNT MY MISCONCEPTION",
                variant="primary",
                size="lg"
            )

    # RESULTS
    gr.Markdown("## 🧠 AI Analysis")

    with gr.Row():

        with gr.Column():

            gr.Markdown("### 🤖 Analysis Agent")

            analysis_output = gr.Textbox(
                label="Detected Reasoning",
                lines=12,
                interactive=False
            )

        with gr.Column():

            gr.Markdown("### 🔍 Verifier Agent")

            verification_output = gr.Textbox(
                label="Verified Finding",
                lines=12,
                interactive=False
            )

    gr.Markdown("## 🎯 Personalized Learning")

    challenge_output = gr.Textbox(
        label="Challenge Generated For You",
        lines=8,
        interactive=False
    )

    # BUTTON ACTION
    analyze_button.click(
        fn=misconception_hunter_ui,
        inputs=[
            question_input,
            answer_input
        ],
        outputs=[
            analysis_output,
            verification_output,
            challenge_output
        ]
    )

    # DEMO EXAMPLE
    gr.Markdown(
        """
        ---
        ### 💡 Try this example

        **Question:**
        Why does a ball continue moving after you stop pushing it?

        **Student Answer:**
        Because the force stays inside the ball and keeps it moving.

        The agents will identify the misconception and generate a new challenge.
        """
    )

    gr.Markdown(
        """
        ---
        <div style="text-align:center">

        **🚀 Team Codenova**

        <b>IMPACTX '26</b>

        *From correcting answers to correcting understanding.*

        </div>
        """
    )

app.launch(share=True)

/tmp/ipykernel_4144/3880913334.py:83: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1ec3afdfbe2c9bedd1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
